In [ ]:
%matplotlib widget

import datetime
import numpy as np
from matplotlib import pyplot
import ipywidgets
import pygetm

In [ ]:
X_EXTENT = 1000.0  # meters
OBC_TYPE = pygetm.FLATHER_TRANSPORT

domain = pygetm.domain.create_cartesian(
    np.linspace(0, X_EXTENT, 51),
    np.linspace(0, 1000.0, 2),
    z0=0.01,
    f=0.0,
    H=50.0,
    interfaces=True,
)
domain.open_boundaries.add_left_boundary(
    "left", 0, 0, domain.ny, OBC_TYPE, 0
)

In [ ]:
sim = pygetm.Simulation(
    domain, runtype=pygetm.RunType.BAROTROPIC_2D, airsea=pygetm.airsea.Fluxes()
)

In [ ]:
sim.T.z.values[...] = np.linspace(0.0, 1.0, 50)
sim.open_boundaries.u.set(0.0)
sim.open_boundaries.v.set(0.0)
sim.open_boundaries.z.set(0.0)

In [ ]:
out = sim.output_manager.add_recorder(interval=1)
out.request("zt", "U")
sim.start(datetime.datetime(2000, 1, 1), domain.maxdt * 0.8, report=100)


In [ ]:
for _ in range(1000):
    sim.advance()

In [ ]:
fig, ax = pyplot.subplots()
(line,) = out.zt[0, 0, :].plot(ax=ax)
ax.set_xlim(0.0, 49.0)
ax.set_ylim(-0.1, 1.0)
ax.grid()


def plot(itime=0):
    line.set_ydata(out.zt[itime, 0, :])
    ax.set_title(f"time = {itime}")


ipywidgets.interact(plot, itime=(0, out.zt.shape[0] - 1))